## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!
            </span>
        </td>
    </tr>
</table>

In [1]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown

In [2]:
load_dotenv(override=True)

True

## OpenAI Hosted Tools

OpenAI Agents SDK includes the following hosted tools:

The `WebSearchTool` lets an agent search the web.  
The `FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
The `ComputerTool` allows automating computer use tasks like taking screenshots and clicking.

### Important note - API charge of WebSearchTool

This is costing me 2.5 cents per call for OpenAI WebSearchTool. That can add up to $2-$3 for the next 2 labs. We'll use free and low cost Search tools with other platforms, so feel free to skip running this if the cost is a concern. Also student Christian W. pointed out that OpenAI can sometimes charge for multiple searches for a single call, so it could sometimes cost more than 2.5 cents per call.

Costs are here: https://platform.openai.com/docs/pricing#web-search

In [3]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

In [5]:
message = "Latest AI Agent typescript frameworks in 2025"

with trace("Search"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

In 2025, several TypeScript frameworks have emerged for developing AI agents:

- **Eliza**: An open-source, Web3-friendly AI agent operating system that integrates seamlessly with blockchain applications. ([arxiv.org](https://arxiv.org/abs/2501.06781?utm_source=openai))

- **BaseAI**: A serverless AI agent framework built with Node.js and TypeScript, enabling the creation of AI agents with memory and tools without managing infrastructure. ([dev.to](https://dev.to/copilotkit/the-tech-stack-for-building-ai-apps-in-2025-12l9?utm_source=openai))

- **Mastra**: A TypeScript framework for building intelligent AI agents capable of executing tasks, accessing knowledge bases, and maintaining persistent memory within workflows. ([sourceforge.net](https://sourceforge.net/software/ai-agents/integrates-with-typescript/?utm_source=openai))

- **VoltAgent**: An open-source TypeScript framework that offers full control and speed for building and orchestrating AI agents, featuring a visual debugging console for step-by-step execution flow inspection. ([dev.to](https://dev.to/voltagent/top-5-ai-agent-frameworks-in-2025-4gab?utm_source=openai))

- **LangGraph**: An open-source framework for AI agents created by LangChain, facilitating the creation, deployment, and management of advanced generative AI workflows with a graph-based architecture. ([antiersolutions.com](https://www.antiersolutions.com/blogs/top-ai-agent-frameworks-to-watch-in-2025-a-complete-guide/?utm_source=openai))

- **LlamaIndex**: An open-source data orchestration framework ideal for developing generative AI and AI agent solutions, featuring ready-to-use agents and tools for context augmentation. ([antiersolutions.com](https://www.antiersolutions.com/blogs/top-ai-agent-frameworks-to-watch-in-2025-a-complete-guide/?utm_source=openai))

- **AgentScope**: A developer-centric framework for building agentic applications, offering unified interfaces and extensible modules for flexible and efficient tool-based agent-environment interactions. ([arxiv.org](https://arxiv.org/abs/2508.16279?utm_source=openai))

These frameworks provide diverse tools and architectures for developing AI agents in TypeScript, catering to various application needs. 

### As always, take a look at the trace

https://platform.openai.com/traces

### We will now use Structured Outputs, and include a description of the fields

In [24]:
# See note above about cost of WebSearchTool

HOW_MANY_SEARCHES = 20

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

# Use Pydantic to define the Schema of our response - this is known as "Structured Outputs"
# With massive thanks to student Wes C. for discovering and fixing a nasty bug with this!

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")


planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)

In [7]:

message = "Latest AI Agent typescript frameworks in 2025"

with trace("Search"):
    result = await Runner.run(planner_agent, message)
    print(result.final_output)

searches=[WebSearchItem(reason='To find the most current frameworks related to AI agent development in TypeScript for 2025.', query='latest AI agent TypeScript frameworks 2025'), WebSearchItem(reason='To identify popular and emerging TypeScript frameworks specifically designed for AI applications in 2025.', query='popular TypeScript AI frameworks 2025'), WebSearchItem(reason='To explore community discussions and expert opinions on the latest TypeScript frameworks for AI agents in 2025.', query='TypeScript frameworks for AI agents discussion 2025')]


In [18]:
@function_tool
def send_email_tool(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("michael@audiowaveai.com") # Change this to your verified email
    to_email = To("michael@audiowaveai.com") # Change this to your email
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    response = sg.client.mail.send.post(request_body=mail)
    print("sent email with status code", response.status_code)
    print(f"""
    From: {from_email.email}
    To: {to_email.email}
    Subject: {subject}
    Body:

    {html_body}
    """)

    # write html_body into a file in sync way
    with open("email.html", "w") as f:
        f.write(html_body)

    return {"status": "success"}

In [ ]:
send_email_tool

In [20]:
INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the
report converted into clean, well presented HTML with an appropriate subject line."""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email_tool],
    model="gpt-4o-mini",
)



In [12]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData,
)

### The next 3 functions will plan and execute the search, using planner_agent and search_agent

In [13]:
async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

### The next 2 functions write a report and email it

In [22]:
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output

async def send_email(report: ReportData):
    """ Use the email agent to send an email with the report """
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return report

### Showtime!

In [25]:
query ="Latest AI Agent typescript frameworks in 2025. Make sure to include code snippets and similarities between the frameworks."

with trace("Research trace"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    await send_email(report)
    print("Hooray!")




Starting research...
Planning searches...
Will perform 19 searches
Searching...
Finished searching
Thinking about report...
Finished writing report
Writing email...
sent email with status code 202

    From: michael@audiowaveai.com
    To: michael@audiowaveai.com
    Subject: Latest AI Agent TypeScript Frameworks in 2025
    Body:

    <h1>Latest AI Agent TypeScript Frameworks in 2025</h1>

<h2>Introduction</h2>
<p>The landscape of AI development continues to evolve rapidly, with TypeScript gaining traction as a preferred language for building AI agents. In 2025, several frameworks have emerged that prioritize efficient development, observability, and integration capabilities. This report explores notable TypeScript-based AI agent frameworks, their features, and provided code snippets to illustrate their usability.</p>

<h2>1. Overview of TypeScript in AI Development</h2>
<p>TypeScript's increasing popularity in the development of AI agents aligns with its static typing advantages that

### As always, take a look at the trace

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">Congratulations on your progress, and a request</h2>
            <span style="color:#00cc00;">You've reached an important moment with the course; you've created a valuable Agent using one of the latest Agent frameworks. You've upskilled, and unlocked new commercial possibilities. Take a moment to celebrate your success!<br/><br/>Something I should ask you -- my editor would smack me if I didn't mention this. If you're able to rate the course on Udemy, I'd be seriously grateful: it's the most important way that Udemy decides whether to show the course to others and it makes a massive difference.<br/><br/>And another reminder to <a href="https://www.linkedin.com/in/eddonner/">connect with me on LinkedIn</a> if you wish! If you wanted to post about your progress on the course, please tag me and I'll weigh in to increase your exposure.
            </span>
        </td>
    </tr>